In [3]:
'''
这是一个图像分类器（image-classification）
流程：1.读取图片数据 2.把图片变为数字 3.训练机器学习模型 4.测试准确率 5.保持模型
预计运行1.5min
'''

import os
import pickle
from skimage.io import imread
from skimage.transform import resize
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV #自动找最优参数
from sklearn.svm import SVC #支持向量机
from sklearn.metrics import accuracy_score #计算准确率
import cv2

# 准备数据
input_dir = 'clf-data'
categories = ['empty', 'not_empty']
data = []
labels = []

for category_idx, category in enumerate(categories):
    for file in os.listdir(os.path.join(input_dir, category)):
        img_path = os.path.join(input_dir, category, file)
        img = imread(img_path)
        img = resize(img, (15, 15))
        data.append(img.flatten()) #15x15 -> 225个数，把二维数组压成一维数组
        labels.append(category_idx)
data = np.asarray(data)
labels = np.asarray(labels)

# 划分训练/测试
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

# 分类器
classifier = SVC()
parameters = [{'gamma': [0.01, 0.001, 0.0001], 'C': [1, 10, 100, 1000]}] #gamma模型复杂度，C控制分类严格程度，共3x4种组合
grid_search = GridSearchCV(classifier, parameters) #自动尝试所有参数组合，并选出最好的组合

# 只用了四行代码就实现了训练过程
grid_search.fit(x_train, y_train)

best_estimator = grid_search.best_estimator_

y_prediction = best_estimator.predict(x_test)

score = accuracy_score(y_prediction, y_test)

print('{}% of samples were correctly classified'.format(str(score * 100)))

pickle.dump(best_estimator, open('./model.p', 'wb'))

100.0% of samples were correctly classified


In [4]:
'''
获取视频中车位的坐标
'''
import cv2

spots = []
def mouse_click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"({x}, {y})")
        spots.append((x, y, 50, 50))  # 默认大小

img = cv2.imread('spot.jpg')
cv2.imshow("Image", img)
cv2.setMouseCallback("Image", mouse_click)

cv2.waitKey(0)

pickle.dump(spots, open('spots.p', 'wb'))

(167, 2)
(166, 6)
(524, 6)
(189, 140)
(528, 138)
(524, 1)
(852, 2)
(529, 134)
(859, 129)
(187, 144)
(527, 139)
(190, 282)
(528, 282)
(528, 139)
(859, 134)
(531, 284)
(859, 281)
(191, 288)
(529, 287)
(196, 433)
(529, 433)
(532, 288)
(859, 283)
(531, 434)
(862, 434)
(195, 435)
(529, 435)
(194, 589)
(529, 581)
(531, 433)
(869, 432)
(537, 581)
(867, 581)
(203, 589)
(534, 585)
(191, 736)
(532, 735)
(532, 586)
(868, 581)
(534, 731)
(871, 733)
(191, 735)
(536, 733)
(200, 887)
(537, 887)
(532, 730)
(869, 728)
(539, 882)
(866, 880)
(197, 887)
(533, 886)
(204, 984)
(537, 985)
(539, 878)
(870, 880)
(538, 984)
(873, 981)


In [4]:
import pickle
import cv2
from skimage.transform import resize

model = pickle.load(open('model.p', 'rb'))
cap = cv2.VideoCapture('data/parking_crop.mp4')
spots = pickle.load(open('spots.p', 'rb'))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    empty_count = 0

    for (x, y, w, h) in spots:

        spot = frame[y:y+h, x:x+w]

        if spot.shape[0] == 0 or spot.shape[1] == 0:
            continue

        spot_resized = resize(spot, (15, 15), anti_aliasing=True)
        spot_flatten = spot_resized.flatten()

        prediction = model.predict([spot_flatten])

        if prediction[0] == 0:
            label = "Empty"
            color = (0, 255, 0)
            empty_count += 1
        else:
            label = "Occupied"
            color = (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label, (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.putText(frame, f"Empty: {empty_count}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (255, 255, 0), 2)

    cv2.imshow("Parking Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()